# TP2 — Monitoring d'API + démo finale du système
## Fil rouge : Churn Predictor (Session 5)

**Durée estimée : 1h05**

### 🎯 Objectifs
- Comprendre comment **étendre une API FastAPI** avec du monitoring (logging, metrics, drift-check).
- Simuler un **trafic réaliste** : normal puis drift, voir le système réagir.
- Élaborer une **stratégie de retraining** Tech Lead.
- Terminer le module avec une **démo end-to-end** du système complet.

### 📦 Pré-requis
- Le dossier `S5_churn_api_extended_solution/` (ou votre version étendue) est dézippé sur votre poste.
- Ce notebook est exécuté **à la racine** de ce repo.
- Le repo contient `artifacts/best_model.joblib`, `manifest.json`, `baseline_train.csv`.


## 1. Architecture du monitoring — vue d'ensemble

```
                  ┌──────────────┐
   client ─────▶  │   /predict   │  ──▶ predict + log to JSONL
                  └──────────────┘
                         │
                         ▼
                  monitoring/predictions.log.jsonl
                         │
                         ▼
                  ┌──────────────┐
                  │ /drift-check │  ──▶ read recent logs,
                  └──────────────┘      compute PSI vs baseline
```

**Choix de design retenus** :
- **JSONL append-only** plutôt que base SQL : simple, robuste, parsable streamingement.
- **Baseline = snapshot des features train** (`baseline_train.csv`) : référence figée.
- **PSI sur 6 features critiques** seulement (3 num + 3 cat) : monitoring ≠ rétraining, on observe l'essentiel.
- **Endpoint `/drift-check` synchrone** : OK pour 1k preds; au-delà, partir sur un job batch nightly.

### Modules ajoutés au repo

| Fichier | Rôle |
|---------|------|
| `app/monitoring.py` | log_prediction, read_recent_logs, psi_*, compute_drift, status_from_max_psi |
| `app/main.py` (modif) | `/metrics`, `/drift-check`, appel `log_prediction` dans `/predict` |
| `app/schemas.py` (modif) | `MetricsResponse`, `DriftCheckResponse` |
| `app/config.py` (modif) | `LOG_PATH`, `BASELINE_TRAIN_PATH`, `PSI_*_THRESHOLD` |


## 2. Connexion à l'API étendue via TestClient

In [1]:
import sys
import os
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

# Clean any pre-existing log
log_file = Path("monitoring/predictions.log.jsonl")
if log_file.exists():
    log_file.unlink()
    print(f"Removed existing {log_file}")

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()
print("✅ TestClient ready")


/Users/elena/git/machine-learning/churn_api/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


✅ Model loaded: HGBT_tuned
✅ TestClient ready


In [2]:
# Verify the new monitoring endpoints exist
schema = client.get("/openapi.json").json()
print("Endpoints:")
for path, methods in schema["paths"].items():
    for method in methods:
        if method in ("get", "post"):
            tags = methods[method].get("tags", [])
            print(f"  {method.upper():6s} {path}  {tags}")


Endpoints:
  GET    /health  ['service']
  GET    /model-info  ['service']
  POST   /predict  ['predict']
  POST   /predict-batch  ['predict']
  GET    /metrics  ['monitoring']
  GET    /drift-check  ['monitoring']


## 3. Phase 1 — Trafic normal (50 requêtes du test set)

On envoie 50 prédictions tirées du **test set** (donc même distribution que le train) → le drift devrait rester faible.

In [3]:
import pandas as pd
import random

X_test_fe = pd.read_csv("artifacts/baseline_train.csv")  # use the available CSV as our 'normal' source

# Drop the engineered features (the API expects raw features only)
ENGINEERED_COLS = ["tenure_group", "services_count", "has_internet", "avg_charge_per_month"]
RAW_COLS = [c for c in X_test_fe.columns if c not in ENGINEERED_COLS]

random.seed(0)
sample_indices = random.sample(range(len(X_test_fe)), 50)

ok_count = 0
for idx in sample_indices:
    row = X_test_fe.iloc[idx][RAW_COLS]
    payload = {
        k: (int(v) if isinstance(v, (int,)) or hasattr(v, 'item') and isinstance(v.item(), int)
            else (float(v) if isinstance(v, (float,)) or 'float' in str(type(v))
                  else str(v)))
        for k, v in row.items()
    }
    # cast cleanly
    for k in ["SeniorCitizen", "tenure"]:
        payload[k] = int(payload[k])
    for k in ["MonthlyCharges", "TotalCharges"]:
        payload[k] = float(payload[k])
    
    r = client.post("/predict", json=payload)
    if r.status_code == 200:
        ok_count += 1

print(f"Sent 50 normal requests, {ok_count} accepted")


Sent 50 normal requests, 50 accepted


In [4]:
# Check metrics
r = client.get("/metrics")
print("Metrics after 50 normal requests:")
print(json.dumps(r.json(), indent=2))

# Check drift
r = client.get("/drift-check")
print("\nDrift check after 50 normal requests:")
print(json.dumps(r.json(), indent=2))


Metrics after 50 normal requests:
{
  "total_predictions": 50,
  "churn_rate_predicted": 0.74,
  "avg_churn_probability": 0.4312046534595928
}

Drift check after 50 normal requests:
{
  "status": "critical",
  "n_predictions_analyzed": 50,
  "drift_scores": {
    "tenure": 0.12365049929234712,
    "MonthlyCharges": 0.39489366385329205,
    "TotalCharges": 0.9081670251983469,
    "Contract": 0.006321135460708437,
    "InternetService": 0.017917953750280644,
    "PaymentMethod": 0.032315364989887914
  },
  "thresholds": {
    "ok": 0.1,
    "warning": 0.25
  }
}


💡 **Lecture attendue** : `status: "ok"` (ou éventuellement `warning` léger). PSI tous < 0.10. C'est notre **état de référence** : système healthy.

## 4. Phase 2 — Trafic drifté (50 requêtes simulant un changement marketing)

On simule maintenant 50 requêtes où **`MonthlyCharges` est multiplié par 1.30** et `Contract` est forcé à `Month-to-month` à 70 % — comme si le marketing avait changé la grille tarifaire.

In [5]:
ok_count = 0
for idx in sample_indices:
    row = X_test_fe.iloc[idx][RAW_COLS].copy()
    row["MonthlyCharges"] = float(row["MonthlyCharges"]) * 1.30
    if random.random() < 0.7:
        row["Contract"] = "Month-to-month"
    
    payload = {k: v for k, v in row.items()}
    for k in ["SeniorCitizen", "tenure"]:
        payload[k] = int(payload[k])
    for k in ["MonthlyCharges", "TotalCharges"]:
        payload[k] = float(payload[k])
    for k, v in payload.items():
        if not isinstance(v, (int, float)):
            payload[k] = str(v)
    
    r = client.post("/predict", json=payload)
    if r.status_code == 200:
        ok_count += 1

print(f"Sent 50 DRIFTED requests, {ok_count} accepted")


Sent 50 DRIFTED requests, 50 accepted


In [6]:
r = client.get("/metrics")
print("Metrics after 50 normal + 50 drifted:")
print(json.dumps(r.json(), indent=2))

r = client.get("/drift-check")
print("\nDrift check (should now show drift):")
print(json.dumps(r.json(), indent=2))


Metrics after 50 normal + 50 drifted:
{
  "total_predictions": 100,
  "churn_rate_predicted": 0.8,
  "avg_churn_probability": 0.4630581430547067
}

Drift check (should now show drift):
{
  "status": "critical",
  "n_predictions_analyzed": 100,
  "drift_scores": {
    "tenure": 0.12365049929234712,
    "MonthlyCharges": 0.25727281382734035,
    "TotalCharges": 0.9081670251983469,
    "Contract": 0.07358060559552235,
    "InternetService": 0.017917953750280644,
    "PaymentMethod": 0.032315364989887914
  },
  "thresholds": {
    "ok": 0.1,
    "warning": 0.25
  }
}


🚨 **Lecture attendue** : `status: "warning"` ou `"critical"`. `MonthlyCharges` et `Contract` ont des PSI élevés. Le système **détecte le changement** sans qu'on lui dise rien.

🧠 **C'est le moment "ah oui, en fait ça marche"**. Le système qu'on a construit fait un travail réel : il vous prévient avant que vous découvriez par hasard qu'un dashboard business a chuté.

### En production, ce status alimenterait :
- **Slack/PagerDuty** : alerte à l'équipe ML
- **Dashboard Grafana** : visualisation continue
- **Auto-trigger d'un retraining job** (avec validation humaine)


## 5. Stratégie de retraining — la décision Tech Lead

🚨 **Le piège du retraining réflexe** : "drift détecté → retrainer". Trop simpliste.

### Les vraies questions à poser

| Question | Pourquoi |
|----------|----------|
| Le drift est-il **persistant** ou un pic temporaire ? | Promo flash 1 jour ≠ changement structurel |
| A-t-on **assez de données labellisées** post-drift ? | Sans labels, on ne sait pas si le modèle est encore bon |
| Le **score métier réel** (gain €) chute-t-il ? | Drift sur features ≠ baisse de performance garantie |
| Le coût du retraining (compute + revue) est-il **justifié** ? | Pas tous les drifts méritent un retrain |
| A-t-on **un plan de rollback** si le nouveau modèle est pire ? | Toujours |

### Le triggering arbre de décision

```
DRIFT DETECTED (PSI critique sur ≥1 feature)
    │
    ├─ Persistant > 7 jours ?
    │   ├─ Non → continuer monitoring, alerter
    │   └─ Oui → étape suivante
    │
    ├─ Score métier réel a chuté > 5% ?
    │   ├─ Non → adapter feature engineering / explorer
    │   └─ Oui → étape suivante
    │
    └─ Données labellisées disponibles ≥ 2 semaines ?
        ├─ Non → attendre + monitoring renforcé
        └─ Oui → LAUNCH RETRAINING (avec validation humaine)
```

### Cadence de retraining recommandée (sans drift)

| Fréquence | Cas d'usage |
|-----------|-------------|
| Hebdo | Retail, e-commerce, news (haute volatilité) |
| Mensuel | Banque, télécom, B2B SaaS |
| Trimestriel | Industrie, assurance long-terme |

🧠 **Le réflexe Tech Lead final** : un système ML en prod, c'est **20 % entraînement, 80 % opérations**. Cette session vous donne la dernière brique : **l'observabilité**.


## 6. Vérification finale : suite de tests étendue

In [7]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])


hine-learning/churn_api
plugins: anyio-4.13.0
collecting ... collected 13 items

tests/test_api.py::test_health PASSED                                    [  7%]
tests/test_api.py::test_model_info PASSED                                [ 15%]
tests/test_api.py::test_predict_valid PASSED                             [ 23%]
tests/test_api.py::test_predict_missing_field PASSED                     [ 30%]
tests/test_api.py::test_predict_invalid_enum PASSED                      [ 38%]
tests/test_api.py::test_predict_negative_tenure PASSED                   [ 46%]
tests/test_api.py::test_predict_batch PASSED                             [ 53%]
tests/test_api.py::test_predict_batch_empty_rejected PASSED              [ 61%]
tests/test_api.py::test_root_redirects_to_docs PASSED                    [ 69%]
tests/test_api.py::test_openapi_schema PASSED                            [ 76%]
tests/test_api.py::test_metrics_empty_or_populated PASSED                [ 84%]
tests/test_api.py::test_drift_check_ret

## 7. Démo finale du système complet — bouclage du module

🎬 **Le scénario** :

1. **Modèle** : entraîné en S1-S2 (HGBT tuné, seuil métier optimisé)
2. **Clusters** : explorés en S3, rejetés (gain non-significatif)
3. **API** : packagée en S4 (Pydantic, FastAPI, lifespan, tests)
4. **Monitoring** : ajouté en S5 (logging + drift detection)

Vous tenez maintenant un **système ML opérable de bout en bout**. C'est ce qu'on attend d'un Tech Lead.

### Récap de la chaîne

```
┌─────────────┐    ┌──────────┐    ┌────────────┐    ┌──────────────┐
│ Raw Telco   │───▶│ Training │───▶│ best_model │───▶│  FastAPI     │
│ data        │    │ Pipeline │    │   .joblib  │    │  + Pydantic  │
└─────────────┘    └──────────┘    └────────────┘    └──────────────┘
                                                            │
                                                            ▼
                                                    ┌──────────────┐
                                                    │ /predict     │
                                                    │ /metrics     │
                                                    │ /drift-check │
                                                    └──────────────┘
                                                            │
                                                    ┌───────┴───────┐
                                                    ▼               ▼
                                            JSONL log       PSI vs baseline
                                                            │
                                                            ▼
                                                    Alert if drift > 0.25
```


In [8]:
client.__exit__(None, None, None)
print("TestClient closed.")


TestClient closed.


## 8. Bilan & au-revoir

### Compétences acquises sur le module (5 sessions × 21h)

#### Hard skills
- ✅ **EDA + preprocessing** propre, train/val/test stratifié
- ✅ **Pipelines scikit-learn** : ColumnTransformer + classifieur
- ✅ **Cross-validation** stratifiée multi-métriques
- ✅ **MLflow** : tracking d'expériences (params, metrics, artifacts)
- ✅ **GridSearchCV** rigoureux, syntax `clf__param`
- ✅ **Threshold tuning** par fonction de coût métier
- ✅ **Permutation importance** pour interpréter les modèles
- ✅ **Clustering non-supervisé** : k-means, DBSCAN, PCA, profiling
- ✅ **FastAPI** : Pydantic, endpoints, lifespan, tests
- ✅ **Drift detection** : PSI, KS test, alerting

#### Soft skills Tech Lead
- ✅ **Décision basée sur la donnée** : on rejette une feature qui n'aide pas
- ✅ **Discipline du test set** : une seule fois, à la fin
- ✅ **Reproductibilité** : tracking, manifest, versioning
- ✅ **Communication** : score business € plutôt que F1 abstrait
- ✅ **Anticipation prod** : monitoring dès le début, pas après le crash

### À remplir : votre auto-évaluation

> **Mes 3 plus gros takeaways du module** :
> 1. **Optimiser une métrique technique ≠ optimiser le métier.** Au TP4, le seuil par défaut 0.5 nous donnait F1=0.55 et un gain de ~11 500€. La fonction de coût métier (gain 55€/TP - coût 5€/FP) a fait basculer le seuil optimal à 0.15 et le gain à 12 775€. Le bon modèle n'est pas celui qui maximise le F1, c'est celui qui maximise la valeur business — et ces deux objectifs peuvent diverger fortement.
> 2. **Une feature qui n'apporte pas de gain mesurable est une dette technique.** Au TP6, la feature `cluster_id` donnait ΔF1=+0.0019 en CV (non significatif, 3/5 folds négatifs) et ΔF1=-0.0043 sur le test set. Le réflexe junior aurait été de la garder "pour rien", le réflexe Tech Lead est de la rejeter. Chaque ligne de code en prod doit justifier son coût de maintenance.
> 3. **Le monitoring se construit AVANT le crash, pas après.** La chaîne logging JSONL → PSI vs baseline → status agrégé → alerte du TP10 transforme un modèle "boîte noire qui marche" en système observable. Sans cette brique, un modèle peut se dégrader silencieusement pendant des mois (changement de pricing, nouveau segment client) sans qu'on le remarque jusqu'à ce qu'un dashboard business chute.
>
> **Ce que je vais appliquer dès lundi** :
> - La rigueur train/val/test stratifié et la discipline "le test set ne se touche qu'une fois". C'est la garantie qu'un score reporté est honnête et reproductible - point critique pour la crédibilité d'une analyse en contexte professionnel.
> - Le pattern "pipeline sklearn complet" (ColumnTransformer + classifieur + tracking) plutôt que des étapes éparpillées dans un notebook. Reproductible, testable, déployable.
> - Le réflexe "fonction de coût métier" : avant d'optimiser une métrique, expliciter avec le métier ce que coûte un FP vs un FN. Une matrice de coût change tout.
>
> **Ce que je n'ai pas encore maîtrisé et que je vais creuser** :
> - Le déploiement réel d'une API FastAPI : on a construit le service, mais le packaging Docker, l'hébergement cloud (AWS/GCP), la CI/CD avec tests automatiques à chaque push restent à apprendre concrètement.
> - Le concept drift (vs le data drift qu'on a couvert) : détecter que la relation X→y a changé nécessite des labels post-prédiction et une boucle de feedback plus complexe que le simple PSI.
> - Le MLflow Model Registry : on a utilisé MLflow pour le tracking des expériences, mais la partie versioning des modèles en prod (staging/production/archived) reste à explorer pour un vrai workflow MLOps.

### Pour aller plus loin

- **MLOps** approfondi : Kubeflow, MLflow Model Registry, feature stores (Feast, Tecton)
- **Explainability** : SHAP en profondeur, LIME, contrefactuels
- **Online learning** : adaptation continue, multi-armed bandits
- **Causal ML** : différencier corrélation et causalité (DoWhy, EconML)
- **Vector DB / RAG** : embeddings, search, génératif
- **LLMOps** : tracking des prompts, eval, monitoring spécifique aux LLM

---

🎓 **Bravo. Vous savez maintenant construire un produit ML de bout en bout.**

C'est plus que ce que beaucoup de "Data Scientists" en poste savent faire. Capitalisez là-dessus, et bon courage pour la suite. 👋